In [3]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model in a `while True` loop and checks each response for any `function_call` items.

- If the model returns a function call, the code runs the tool, appends the tool output to the message history, and loops again.
- If the model returns only a normal message and no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the response**.


In [4]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Overriding of current TracerProvider is not allowed


In [5]:
with tracer.start_as_current_span("my_operation") as span:
    query = "How does the agentic loop keep calling the model until it stops?"
    answer = rag.rag(query)
    # print(answer)

{
    "name": "my_operation",
    "context": {
        "trace_id": "0x87a4e04cf153c879b7a9146de55ad5ed",
        "span_id": "0x93f6ea6025ebdd59",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-27T21:52:51.309650Z",
    "end_time": "2026-07-27T21:52:53.134439Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "d2a307cf-5189-474b-a6b3-daf3ce261a3b",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


In [4]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    def search(self, query):
        with tracer.start_as_current_span("search") as search_span:
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as llm_span:
            return super().llm(prompt)

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            search_results = self.search(query)

            prompt = self.build_prompt(query, search_results)

            response = self.llm(prompt)

        return response

In [6]:
## Q1

from openai import OpenAI

from gitsource import GithubRepositoryDataReader
from minsearch import Index

from rag_helper import RAGBase

from dotenv import load_dotenv
load_dotenv()

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()
rag = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
response = rag.rag(query)

In [17]:
response.usage

ResponseUsage(input_tokens=7111, input_tokens_details=InputTokensDetails(cached_tokens=6912, cache_write_tokens=0), output_tokens=109, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7220)

In [5]:
class RAGTraced(RAGBase):
    def search(self, query):
        with tracer.start_as_current_span("search") as search_span:
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as llm_span:
            return super().llm(prompt)

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            search_results = self.search(query)

            prompt = self.build_prompt(query, search_results)

            response = self.llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)

        return response

In [ ]:
## Q2/Q3
rag = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
response = rag.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x005d962eff649549d7a2804cc0cdcc2d",
        "span_id": "0xccb2982edb5648a2",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x687541e9b467dd26",
    "start_time": "2026-07-27T22:29:03.670404Z",
    "end_time": "2026-07-27T22:29:03.673845Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "d2a307cf-5189-474b-a6b3-daf3ce261a3b",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x005d962eff649549d7a2804cc0cdcc2d",
        "span_id": "0x419843a7330440fb",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [1]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [20]:
rag = RAGTraced(index=index, llm_client=client)

query = "How does the agentic loop keep calling the model until it stops?"
response = rag.rag(query)

In [21]:
## Q4/Q5/Q6
import sqlite3
conn = sqlite3.connect("traces.db")
for row in conn.execute("SELECT name, start_time, end_time, end_time - start_time AS duration_ns, (end_time - start_time) / 1000000, input_tokens FROM spans"):
    print(row)
conn.close()

('search', 1785192361329072980, 1785192361330632191, 1559211, 1, None)
('llm', 1785192361333009703, 1785192364122708360, 2789698657, 2789, None)
('rag', 1785192361329014815, 1785192364125008903, 2795994088, 2795, 7111)
('search', 1785192471031348579, 1785192471035720868, 4372289, 4, None)
('llm', 1785192471038890889, 1785192473235853103, 2196962214, 2196, None)
('rag', 1785192471031290946, 1785192473238758312, 2207467366, 2207, 7111)
('search', 1785192474439870184, 1785192474444390242, 4520058, 4, None)
('llm', 1785192474451420204, 1785192477259236889, 2807816685, 2807, None)
('rag', 1785192474439815942, 1785192477261622704, 2821806762, 2821, 7111)
('search', 1785192478432726807, 1785192478435306373, 2579566, 2, None)
('llm', 1785192478445533882, 1785192480150015073, 1704481191, 1704, None)
('rag', 1785192478432689181, 1785192480152870618, 1720181437, 1720, 7111)
('search', 1785192569482414991, 1785192569493481628, 11066637, 11, None)
('llm', 1785192569512348409, 1785192571133490133, 1

In [23]:
conn = sqlite3.connect("traces.db")
for row in conn.execute("SELECT AVG((end_time - start_time) / 1000000) FROM spans WHERE name='llm'"):
    print(row)
conn.close()

(2299.5555555555557,)
